In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import ssl
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    """实验超参数配置"""
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数 (适当增加维度以应对 Junyi 的复杂性)
        self.embed_dim = 128
        self.seq_len = 100
        self.tcan_layers = 3 # 增加层数以提取深层时序特征
        self.kernel_size = 3
        self.dropout = 0.4

        # 训练参数
        self.batch_size = 64
        self.epochs = 20 # 增加训练轮数
        self.lr = 0.0005 # 适当降低学习率以稳定收敛
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data_junyi"
        self.dataset_name = "junyi"
        self.fallback_url = "https://raw.githubusercontent.com/bigdata-ustc/EduData/master/static/junyi/junyi_problem_log.csv"

# ==========================================
# 2. 数据处理模块 (Data Processing)
# ==========================================

class JunyiDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]
        seq_len = len(q_seq)

        if seq_len >= self.max_len:
            q_seq, r_seq = q_seq[-self.max_len:], r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessorJunyi:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def download_data(self):
        """自动安装 EduData 并通过名称或 URL 下载数据集"""
        csv_file = self._find_csv(self.config.data_dir)
        if csv_file:
            print(f"发现本地 Junyi 数据: {csv_file}")
            return csv_file

        print("正在安装/检查 EduData...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])

        from EduData import get_data
        try:
            print(f"尝试通过 EduData 下载: {self.config.dataset_name}")
            get_data(self.config.dataset_name, self.config.data_dir)
        except Exception as e:
            print(f"EduData 下载失败 ({e})，尝试使用备用链接下载...")
            save_path = os.path.join(self.config.data_dir, "junyi_problem_log.csv")
            context = ssl._create_unverified_context()
            try:
                req = urllib.request.Request(self.fallback_url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, context=context) as response, open(save_path, 'wb') as out_file:
                    out_file.write(response.read())
            except Exception as e2:
                raise RuntimeError(f"所有下载尝试均失败: {e2}")

        return self._find_csv(self.config.data_dir)

    def _find_csv(self, root_dir):
        log_files = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith(".csv"):
                    full_path = os.path.join(root, file)
                    if "log" in file.lower(): return full_path
                    log_files.append(full_path)
        return log_files[0] if log_files else None

    def process(self, order_strategy="chronological"):
        file_path = self.download_data()

        print("正在加载数据并自动检测字段...")
        try:
            # 增加读取量到 1,000,000 行以获得更多特征，但仍保持处理速度
            df = pd.read_csv(file_path, low_memory=False, nrows=1000000)
        except Exception:
            df = pd.read_csv(file_path, low_memory=False, nrows=1000000, encoding='latin-1')

        cols = df.columns.tolist()
        user_col = next((c for c in cols if c.lower() in ['user_id', 'uid', 'user']), None)
        item_col = next((c for c in cols if c.lower() in ['exercise_name', 'exercise', 'problem_id', 'item_id']), None)
        correct_col = next((c for c in cols if c.lower() in ['correct', 'is_correct']), None)
        time_col = next((c for c in cols if c.lower() in ['time_done', 'timestamp', 'time', 'done_time']), None)

        # 数据清洗
        df.dropna(subset=[user_col, item_col, correct_col], inplace=True)
        df[correct_col] = (df[correct_col] > 0).astype(int)

        # 映射 ID
        user_map = {val: i+1 for i, val in enumerate(df[user_col].unique())}
        prob_map = {val: i+1 for i, val in enumerate(df[item_col].unique())}

        df['user_id_mapped'] = df[user_col].map(user_map)
        df['item_id'] = df[item_col].map(prob_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(prob_map) + 1
        # 即使没有 Skill Map，我们也假设题目 ID 就是其 Concept 基础
        self.config.num_concepts = self.config.num_questions

        # 难度系数
        item_difficulty = df.groupby('item_id')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, acc in item_difficulty.items():
            diff_values[iid] = 1.0 - acc

        q_k_relation = np.eye(self.config.num_questions)

        # 选取 1500 名用户构造子集以增加数据多样性
        all_q, all_r, all_s = [], [], []
        target_users = df['user_id_mapped'].unique()[:1500]
        df_sub = df[df['user_id_mapped'].isin(target_users)]

        for uid, group in tqdm(df_sub.groupby('user_id_mapped'), desc=f"消融策略: {order_strategy}"):
            if len(group) < 5: continue

            if order_strategy == "chronological":
                if time_col: group = group.sort_values(by=time_col)
            elif order_strategy == "reverse":
                if time_col: group = group.sort_values(by=time_col, ascending=False)
            elif order_strategy == "random":
                group = group.sample(frac=1, random_state=42)
            elif order_strategy == "skill_grouped":
                # 在 Junyi 中，按题目名称排序通常能把相似类型的题目聚在一起
                sort_cols = [item_col]
                if time_col: sort_cols.append(time_col)
                group = group.sort_values(by=sort_cols)

            all_q.append(group['item_id'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 模型定义 (TCAN Architecture)
# ==========================================

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q, K, V = self.query(x), self.key(x), self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        conv_out = self.conv(z.permute(0, 2, 1))[:, :, :z.size(1)].permute(0, 2, 1)
        return self.dropout(self.relu(self.norm(conv_out + residual)))

class HIG_TCAN_CD_Model(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD_Model, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))

        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)

        # 难度信息投影
        self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Sequential(
            nn.Linear(1, config.embed_dim // 2),
            nn.Tanh(),
            nn.Linear(config.embed_dim // 2, config.embed_dim)
        )

        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])

        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 128),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        # 结合题目内容和难度
        q_feat = self.question_emb(q_ids)
        d_val = self.diff_values[q_ids].unsqueeze(-1)
        d_feat = self.diff_proj(d_val)

        q_emb = q_feat + d_feat

        x = self.input_proj(torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1))
        h = x
        for layer in self.tcan_layers: h = layer(h)

        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

# ==========================================
# 4. 训练与实验运行
# ==========================================

def train_and_eval(config, data_bundle):
    train_loader, test_loader, q_k_rel, diff_values = data_bundle
    model = HIG_TCAN_CD_Model(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=1e-5)

    # 学习率调度器，帮助模型在后期收敛
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    best_auc = 0
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        for q, r, mask, s in train_loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h_seq, q_seq_emb, s_static = model(q, r, s)
            feat = torch.cat([h_seq[:, :-1, :], q_seq_emb[:, 1:, :], s_static[:, 1:, :]], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            # 只计算有效 mask 位置的损失
            loss = (F.binary_cross_entropy(preds, r[:, 1:], reduction='none') * mask[:, 1:]).sum() / (mask[:, 1:].sum() + 1e-8)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        model.eval()
        all_p, all_t = [], []
        with torch.no_grad():
            for q, r, mask, s in test_loader:
                q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
                h, qe, ss = model(q, r, s)
                feat = torch.cat([h[:, :-1, :], qe[:, 1:, :], ss[:, 1:, :]], dim=-1)
                p = model.pred_mlp(feat).squeeze(-1)
                m = mask[:, 1:].bool()
                all_p.extend(p[m].cpu().numpy())
                all_t.extend(r[:, 1:][m].cpu().numpy())

        auc = roc_auc_score(all_t, all_p)
        if auc > best_auc: best_auc = auc

    return best_auc

def run_experiment():
    config = Config()
    processor = DataProcessorJunyi(config)
    results = {}

    strategies = ["chronological", "reverse", "random", "skill_grouped"]

    for strategy in strategies:
        print(f"\n[实验开始] 策略类型: {strategy}")
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process(order_strategy=strategy)

        t_q, v_q, t_r, v_r, t_s, v_s = train_test_split(q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42)

        train_loader = DataLoader(JunyiDataset(t_q, t_r, t_s, config.seq_len), batch_size=config.batch_size, shuffle=True)
        test_loader = DataLoader(JunyiDataset(v_q, v_r, v_s, config.seq_len), batch_size=config.batch_size)

        best_auc = train_and_eval(config, (train_loader, test_loader, q_k_rel, diff_values))
        results[strategy] = best_auc
        print(f"--- 策略 {strategy} 完成: Best AUC = {best_auc:.4f} ---")

    print("\n" + "="*50)
    print("Junyi 数据集消融实验汇总表 (优化版)")
    print("-" * 50)
    for k in sorted(results, key=results.get, reverse=True):
        print(f"{k:<25} | AUC: {results[k]:.4f}")
    print("="*50)

if __name__ == "__main__":
    run_experiment()


[实验开始] 策略类型: chronological
正在安装/检查 EduData...


downloader, INFO http://base.ustc.edu.cn/data/JunyiAcademy_Math_Practicing_Log/junyi.rar is saved as data_junyi/junyi.rar


尝试通过 EduData 下载: junyi

downloader, INFO data_junyi/junyi.rar is unrar to data_junyi/junyi



正在加载数据并自动检测字段...


...

--- 策略 reverse 完成: Best AUC = 0.6848 ---

[实验开始] 策略类型: random
发现本地 Junyi 数据: ./data_junyi/junyi/junyi_ProblemLog_original.csv
正在加载数据并自动检测字段...


消融策略: random: 100%|██████████| 1500/1500 [00:00<00:00, 1761.33it/s]


--- 策略 random 完成: Best AUC = 0.6865 ---

[实验开始] 策略类型: skill_grouped
发现本地 Junyi 数据: ./data_junyi/junyi/junyi_ProblemLog_original.csv
正在加载数据并自动检测字段...


消融策略: skill_grouped: 100%|██████████| 1500/1500 [00:01<00:00, 938.97it/s] 


--- 策略 skill_grouped 完成: Best AUC = 0.6916 ---

Junyi 数据集消融实验汇总表 (优化版)
--------------------------------------------------
skill_grouped             | AUC: 0.6916
chronological             | AUC: 0.6867
random                    | AUC: 0.6865
reverse                   | AUC: 0.6848
